# Your RAG Score Is Hiding Which Half Is Broken

*Split retrieval quality from generation quality in DeepEval and find out which component is costing you accuracy.*

**Read this if**

- Your RAG app has one quality number and you cannot tell whether to fix retrieval or the prompt.
- Your faithfulness score looks fine while users report wrong answers.
- You want to watch two real failures scored by five metrics, and see which metric noticed.

The previous post left Halcyon with a shortlist and three metrics on a reserved list, waiting for a retriever. This post builds the retriever. The three come off the shelf. One of them turns out to be the only metric in the building that can see what goes wrong.

Halcyon is a payments API vendor with about two hundred staff. Its customers are developers integrating card payments, usually under deadline, usually at night. Its documentation is public and date-versioned. Six months ago the docs team put an assistant on the site. A developer types a question, a search step finds a few pages, and a language model writes an answer from them. The developer reads the answer and writes code against it. Nobody at Halcyon reads the answer. The code moves money.

In March, Halcyon changed how it signs webhooks. The old scheme put an HMAC-SHA1 digest, a keyed hash, in an X-Halcyon-Signature header. The new one uses HMAC-SHA256 with a timestamp. The old page is still published, because accounts pinned to older API versions still receive the old header. Its last sentence says so. The assistant's dashboard shows one quality number, faithfulness. It has been green since launch. Last week a developer asked it how to verify a webhook signature and shipped what it told her. On Monday she opened a ticket. Every event was failing. The docs team pulled up the dashboard. Still green.

## One number cannot say which half to fix

RAG stands for retrieval augmented generation. A retriever searches a store of text and returns the pieces most like the question. A generator, the language model, writes an answer from those pieces. Two components, one output. When the output is wrong, the one number on the dashboard cannot say which of the two to fix. The docs team's green light is a score for the generator. Nobody was scoring the retriever at all.

An integration test that fails says the pipeline is broken, and that is all it says. A unit test per stage says which stage. That split is the idea DeepEval's RAG guide is built around. There is one difference from real unit tests. These have a judge where a unit test has a fixed expected value. A judge is a second language model that applies a fixed rubric to an output and returns a score with a reason. DeepEval's five RAG metrics sit on either side of the component line. Three read what the retriever returned. Two read what the generator wrote.

Each metric reads a test case. For this assistant that is the question, the answer, the chunks the retriever returned, and a reference answer we wrote by hand. The reference lives in a golden, the record we keep per question regardless of any run. A metric that needs the reference is reference-based. One that reads only the question, the answer and the chunks is referenceless. That distinction turns out to matter more than the retriever-versus-generator one. A threshold is the minimum score that counts as a pass, and every DeepEval score reads the same way. Higher is better. The threshold is a floor.

Here is the plan. We build the assistant as a graph with a retrieve node and a generate node. The two halves are then visible in the code as well as in the metrics. We score one question with all five metrics and open every verdict, the yes or no the judge returns for each chunk or sentence. We break the retriever on purpose and watch which scores move. Then we fire the two traps this corpus was built around, and find out which metric saw which. We close by moving retrieval into a real vector database, because that is where the fix lives.

The notebook reads `GEMINI_API_KEY` from the environment. A full run costs a little under two dollars, and the exact bill is printed at the end.

In [1]:
# Shared Halcyon scaffolding, built in post 0 and re-created verbatim in every later post.
# !uv pip install deepeval==4.2.1 langgraph==1.2.11 langchain-core==1.6.2 langchain-google-genai==4.4.0 google-genai==2.22.0
import os, re, json, time, hashlib, logging, statistics
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version
from typing import TypedDict

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from deepeval.dataset import Golden

logging.getLogger("google_genai.models").setLevel(logging.ERROR)   # silence an SDK advisory about tool calling

if not os.environ.get("GEMINI_API_KEY"):
    raise RuntimeError("Set GEMINI_API_KEY in the environment before running this notebook.")

JUDGE_MODEL = "gemini-3.6-flash"      # the judge every later post uses; this post never calls it
APP_MODEL = "gemini-3.6-flash"        # the model inside the assistant under test
MAX_CONCURRENCY = 4                   # model calls in flight at any moment
PRICE_IN, PRICE_OUT = 0.75 / 1e6, 3.75 / 1e6   # USD per token, gemini-3.6-flash standard tier, 2026 rate
T0 = time.time()
USAGE = []                            # one usage_metadata dict per model call, for the cost ledger


def with_retry(fn, *args, attempts=5, base_delay=2.0):
    """Call fn(*args); on a 429 or 503 wait 2, 4, 8, 16 seconds and try again."""
    for attempt in range(attempts):
        try:
            return fn(*args)
        except Exception as exc:
            transient = any(code in str(exc) for code in ("429", "503", "RESOURCE_EXHAUSTED"))
            if not transient or attempt == attempts - 1:
                raise
            time.sleep(base_delay * 2 ** attempt)


# No temperature, top_p or top_k: Google lists them as deprecated sampling parameters to strip from requests.
llm = ChatGoogleGenerativeAI(model=APP_MODEL, google_api_key=os.environ["GEMINI_API_KEY"], max_retries=0)

DOCS = [dict(zip(('slug', 'title', 'status', 'updated', 'body'), row)) for row in [
    ("quickstart", "Quickstart", "current", "2026-03-01", "Create an account, then generate a test API key from the dashboard; test keys start with sk_test_ and live keys with sk_live_. Install the SDK with pip install halcyon (the current SDK is 3.x). Create your first charge with POST /v1/charges, passing amount in minor units (an integer: 1999 means 19.99), a lowercase ISO 4217 currency such as usd or eur, and a source token from Halcyon.js. Send the Halcyon-Version header with the API version you tested against so later changes do not reach you until you opt in. Use test card 4242 4242 4242 4242 for a successful payment and 4000 0000 0000 0002 for a decline."),
    ("api-conventions", "API conventions", "current", "2026-03-01", "The API is served over HTTPS at https://api.halcyon.dev and is date-versioned: send Halcyon-Version: 2026-03-01 (or an earlier version) with each request; requests without the header use the account's default version. Amounts are integers in minor units and currencies are lowercase ISO 4217 codes. Every POST to a money-moving endpoint (charges, refunds, payouts) must include an Idempotency-Key header, a client-generated unique string; a UUID v4 is recommended. Replaying a request with the same key within 24 hours returns the original response instead of moving money again. Requests without the header are accepted for backward compatibility, so a retried POST without one creates a second charge. Keys are scoped to the account and to the endpoint."),
    ("authentication", "Authentication", "current", "2026-01-15", "Authenticate every request with an Authorization: Bearer header carrying your secret key. Test-mode keys (sk_test_) never move real money; live-mode keys (sk_live_) do. Restricted keys can be limited to read-only access or to specific resources. Rotate a key from the dashboard: the new key is active immediately and the old key keeps working for 24 hours so deployments can roll over without downtime; revoke it sooner from the same screen if it was exposed. A missing or invalid key returns HTTP 401 with type authentication_error. Never send secret keys from browsers or mobile apps; use Halcyon.js publishable keys (pk_) there."),
    ("errors", "Errors", "current", "2026-03-01", "Errors return a JSON body with type, code, message and request_id. Types map to HTTP status: invalid_request_error (400), authentication_error (401), card_error (402), rate_limit_error (429) and api_error (500 to 504). card_error codes include card_declined, insufficient_funds, expired_card and incorrect_cvc; show the cardholder a generic message and let them try another card. Requests that return 5xx may have partially completed: retry them with exponential backoff, and quote the request_id when you contact support. Do not retry 4xx errors other than 429."),
    ("rate-limits", "Rate limits", "current", "2025-11-01", "Live mode allows 100 requests per second per account with short bursts to 200; test mode allows 25 requests per second. Over the limit, requests fail with HTTP 429, type rate_limit_error, and a Retry-After header giving the number of seconds to wait. Back off exponentially with jitter rather than retrying in a tight loop, and spread bulk work such as migrations or invoice runs over time or through a queue. Webhook deliveries to your endpoint do not count against the limit. Contact support with your expected peak if you need a higher limit."),
    ("charges-create", "Create a charge", "current", "2026-03-01", "POST /v1/charges creates a charge. Required parameters: amount (integer, minor units), currency (lowercase ISO 4217) and source (a token from Halcyon.js or a saved payment method id starting pm_). Optional: description, metadata (up to 20 key-value pairs), and capture, which defaults to true; pass capture=false to authorize only, then capture within 7 days with POST /v1/charges/{id}/capture. The response is a charge object with id (ch_), status (succeeded, pending or failed), amount, currency, amount_refunded and failure_code. Card declines return HTTP 402 with a card_error body; a charge that could not be attempted at all returns 400."),
    ("refunds", "Refunds", "current", "2026-02-10", "POST /v1/refunds with charge (a ch_ id) refunds a charge in full; pass amount to refund part of it. A charge can be refunded several times until the refunded total reaches the original amount; a request beyond the remaining balance returns 400 with code amount_exceeds_refundable. Refunds of a pending charge are rejected until the charge succeeds. The refund object has id (re_), status (pending, succeeded or failed) and amount. Refunds settle to the cardholder in 5 to 10 business days depending on the issuing bank; status succeeded means Halcyon has released the funds, not that the cardholder has seen them yet."),
    ("webhooks-overview", "Webhooks", "current", "2026-03-01", "Halcyon notifies your endpoint about events by POSTing a JSON event object with id (evt_), type, created and data. Event types include charge.succeeded, charge.failed, refund.succeeded and dispute.opened. Your endpoint must return a 2xx within 10 seconds; otherwise the delivery is retried with exponential backoff (1 minute, 5 minutes, 30 minutes, 2 hours, then every 6 hours) for up to 72 hours, after which the event is marked failed and shown in the dashboard. Delivery is at-least-once: the same event can arrive more than once and events can arrive out of order, so store the event id and ignore repeats. Always verify the signature header before trusting an event; see the webhook signatures pages."),
    ("webhook-signatures-v1", "Webhook signatures (v1)", "deprecated", "2026-03-01", "Each webhook request carries an X-Halcyon-Signature header containing a hex-encoded HMAC-SHA1 of the raw request body, keyed with your endpoint's signing secret (whsec_). To verify, compute HMAC-SHA1 over the exact bytes you received, before any JSON parsing, and compare with the header using a constant-time comparison. Reject the event if the values differ. Do not re-serialize the JSON before hashing, since key ordering and whitespace change the digest. This scheme was deprecated on 2026-03-01 in favour of webhook-signatures-v2, which uses HMAC-SHA256 with a timestamped Halcyon-Signature header; it remains in service for accounts pinned to API versions before 2026-03-01."),
    ("webhook-signatures-v2", "Webhook signatures (v2)", "current", "2026-03-01", "Each webhook request carries a Halcyon-Signature header of the form t=<unix timestamp>,v2=<hex digest>. The signed payload is the timestamp, a period, and the raw request body. To verify, compute HMAC-SHA256 of that payload with your endpoint's signing secret (whsec_), compare with the v2 value using a constant-time comparison, and reject the event if the timestamp is more than 300 seconds old, which defeats replay of captured requests. During a signing secret rotation the header carries two v2 values for 24 hours; accept the event if either matches. This is the default scheme for API version 2026-03-01 and later, and the only scheme SDK 3.x verifies."),
    ("changelog-2026-03", "Changelog: API version 2026-03-01", "current", "2026-03-01", "API version 2026-03-01. Webhook signatures v2 (HMAC-SHA256, timestamped Halcyon-Signature header) are now the default; v1 (HMAC-SHA1, X-Halcyon-Signature) is deprecated and is sent only to accounts pinned to earlier versions. Halcyon SDK 3.0 ships with v2 verification helpers and no longer verifies v1 headers; SDK 2.x continues to verify v1 only. The Idempotency-Key replay window is extended from 1 hour to 24 hours, and reusing a key with a different request body now returns 400 with code idempotency_key_reused instead of silently returning the earlier response. Request and response shapes for charges and refunds are unchanged."),
    ("support-sla", "Support and SLA", "current", "2026-01-15", "Support is available to all accounts through the dashboard and support@halcyon.dev. Tickets are triaged into three priorities. P1: any report that funds have been affected, including duplicate or double charges, charges of the wrong amount, and refunds or payouts that did not arrive, regardless of how the report is worded or whether the ticket is mainly about something else; first response within 1 hour, 24 hours a day. P2: a live-mode integration that is failing, such as authentication errors, webhook deliveries or signature verification failing, or unexpected 4xx or 5xx responses from live endpoints; first response within 4 business hours. P3: how-to questions, documentation gaps, feature requests and anything in test mode; first response within 1 business day."),
]]

TICKETS = [dict(zip(('id', 'submitted', 'subject', 'body'), row)) for row in [
    ("T-1001", "2026-08-03", "Which signature header am I supposed to verify?", "We're integrating webhooks and every event we receive has an X-Halcyon-Signature header but no Halcyon-Signature header, so the verify() helper in SDK 3.1 throws 'no v2 signature present'. The webhook overview says to verify signatures and links to two pages. Our account was created in 2025 and I think we're pinned to 2025-11-01. Is v1 still supported, do we need to move to v2, and how do we change the API version without breaking our existing charge flow? This is blocking our launch on the live account."),
    ("T-1002", "2026-08-04", "429s in test mode when our integration suite runs", "Our CI runs about 40 requests per second against test mode at peak and we get bursts of rate_limit_error with Retry-After: 1. Live mode is fine. Is the test-mode limit documented anywhere, and can it be raised for CI? We can add a throttle but it makes the suite roughly three times slower."),
    ("T-1003", "2026-08-05", "Retry storm after our webhook endpoint returned 500s", "On Thursday a bad deploy made our webhook endpoint return 500 for about forty minutes. Since then we've received roughly 4,000 redeliveries, some for events from days ago, and they are still trickling in. Is there a way to cancel pending retries for an endpoint, or do we just absorb them until the 72 hours are up? Probably unrelated, but while going through the logs we found a few orders from that afternoon that were charged twice, two charge ids each; we assume our order service retried the charge call on timeout and we'll refund them on our side. Mainly we need the retries to stop."),
    ("T-1004", "2026-08-06", "Getting 400 idempotency_key_reused on retries", "Since we moved to API version 2026-03-01 last week, our retry path fails with 400 idempotency_key_reused. We generate one Idempotency-Key per order and reuse it if the first attempt fails, which used to work. Looking closer, our retry adds a metadata.attempt field to the body. Is the body compared exactly? What is the recommended pattern for retries where the body legitimately changes? This is live traffic; failed retries currently land in a dead-letter queue and someone replays them by hand."),
    ("T-1005", "2026-08-07", "Partial refund returns amount_exceeds_refundable", "A customer paid 120.00 EUR and we refunded 80.00 EUR last week. Refunding the remaining 40.00 EUR today returns 400 amount_exceeds_refundable. It turns out our cancellation flow had already issued a separate 40.00 EUR refund that nobody noticed, so the error is correct. What we actually need: is there a field on the charge that tells us the remaining refundable balance, so we can check before calling refunds? And is there any per-month cap on refunds?"),
    ("T-1006", "2026-08-10", "SDK 3.0 upgrade broke webhook verification, rolled back", "We upgraded from SDK 2.9 to 3.0 on Monday. From the first deploy every webhook failed verification with 'no v2 signature present' and our fulfilment queue stalled. We rolled back to 2.9 after about 25 minutes and verification works again. Our account is on API version 2025-11-01. What exactly changed in 3.0, and what is the migration order: SDK first or API version first? One side effect: while the queue was stalled a colleague re-ran the stuck jobs by hand and two customers were charged a second time; we've refunded one and are chasing the other. The verification question is the one we need answered before we try again."),
    ("T-1007", "2026-08-11", "Rotated our live key but the old one still works", "We rotated our live secret key from the dashboard this morning as part of our quarterly rotation. Two hours later, requests using the old key still succeed. Is rotation delayed, or did it not take? We expected the old key to stop working straight away. Also, is there an audit log showing which key made which request?"),
    ("T-1008", "2026-08-12", "How do we simulate a dispute in test mode?", "We're building our dispute.opened handler and can't find a way to trigger a dispute against a test-mode charge. Is there a test card number or a dashboard button that opens a dispute? Also, what fields does the dispute object carry? Not urgent, we're a few weeks from launch."),
    ("T-1009", "2026-08-13", "Batch import script times out around row 200", "We're migrating 1,800 customers from our old provider. The script imports each saved payment method and immediately creates the first invoice charge. It runs fine for about 200 rows and then requests start timing out at 30 seconds; the script retries each failed row up to three times and eventually finishes, but the whole run takes hours. Is there a batch endpoint, or a recommended concurrency for imports? Somewhat related: our finance team says a dozen or so of the imported customers have two invoice charges instead of one, which we think is the retry path. We can clean that up ourselves once the import is stable."),
    ("T-1010", "2026-08-14", "Duplicate webhook deliveries and events arriving out of order", "We're seeing the same evt_ id delivered two or three times, sometimes minutes apart, and occasionally a refund.succeeded arrives before the charge.succeeded for the same order. No money problem that we can see, every charge is correct in the dashboard, but our handler assumed one delivery per event and now double-writes some rows. Is this expected behaviour? Should we be deduplicating on our side and, if so, on which field?"),
    ("T-1011", "2026-08-17", "Intermittent 502s from POST /v1/charges since Tuesday", "Roughly one in fifty POST /v1/charges calls returns a 502 with an HTML body rather than your JSON error format, all from the eu-west region. Our client retries 5xx with backoff as your errors page recommends and the retry almost always succeeds, so checkout is mostly working. Request ids for six failures are attached. Two customers have emailed our support saying their statement shows the same order twice, which I assume is the retry; we'll sort that out with them. The 502s are what we need you to look at."),
    ("T-1012", "2026-08-18", "Do you support Apple Pay and Google Pay?", "Our product team wants wallet payments at checkout. I can't find Apple Pay or Google Pay anywhere in the docs. Is it supported through Halcyon.js, on a roadmap, or would we need a second provider for wallets? Also, is there a fee difference for wallet payments?"),
]]

def corpus_fingerprint(docs, tickets):
    return hashlib.sha256(json.dumps([docs, tickets], sort_keys=True).encode()).hexdigest()[:12]

SUMMARIZE = ("You write one-line summaries for Halcyon's support triage queue. Summarize the ticket below "
             "in one sentence of at most 25 words, in plain English.\n\nTicket:\n{ticket}")

def ticket_text(ticket):
    return f"Subject: {ticket['subject']}\n\n{ticket['body']}"

class AppState(TypedDict):
    question: str
    context: list[str] | None
    answer: str

SYSTEM = ("You are Halcyon's developer support assistant. Halcyon is a payments API vendor. When "
          "documentation pages are supplied, answer from them alone and say so if they do not cover the "
          "question. Be concise and specific.")

def respond(state: AppState) -> dict:
    prompt = state["question"]
    if state.get("context"):
        prompt = "Documentation pages:\n\n" + "\n\n".join(state["context"]) + f"\n\nQuestion: {prompt}"
    reply = with_retry(llm.invoke, [SystemMessage(SYSTEM), HumanMessage(prompt)])
    USAGE.append(reply.usage_metadata or {})
    return {"answer": reply.text.strip()}

builder = StateGraph(AppState)
builder.add_node("respond", respond)
builder.add_edge(START, "respond")
builder.add_edge("respond", END)
APP = builder.compile()

def answer(question, context=None):
    return APP.invoke({"question": question, "context": context})["answer"]

SLA = next(d for d in DOCS if d["slug"] == "support-sla")["body"]
CLASSIFY = ("Assign a support priority to the ticket summary below using Halcyon's SLA. Reply with exactly "
            "one of P1, P2 or P3 and nothing else.\n\nSLA:\n{sla}\n\nTicket summary: {summary}")

def triage(ticket):
    """Ticket in, one-sentence summary plus severity out. The severity step sees only the summary."""
    summary = answer(SUMMARIZE.format(ticket=ticket_text(ticket)))
    verdict = re.search(r"P[123]", answer(CLASSIFY.format(sla=SLA, summary=summary)))
    return {"id": ticket["id"], "summary": summary, "severity": verdict.group(0) if verdict else "unparsed"}

GOLDEN_SPEC = {  # severity, trap, pages, reference summary, why it exists
    "T-1001": ("P2", "two-truths", ["webhook-signatures-v1", "webhook-signatures-v2", "changelog-2026-03", "api-conventions"], "Live webhooks carry only the v1 X-Halcyon-Signature header, so SDK 3.1 verification fails; asks whether to move the account to API version 2026-03-01.", "Both signature pages are published and both are correct for someone; the right answer depends on the account's API version."),
    "T-1002": ("P3", None, ["rate-limits"], "CI hits test-mode rate limits at 40 requests per second; asks whether the limit is documented or can be raised.", "A plain how-to with one page that answers it fully; the control case."),
    "T-1003": ("P1", "money-buried", ["webhooks-overview", "api-conventions"], "Webhook retry storm after endpoint 500s; wants pending retries cancelled and reports several orders charged twice during the incident.", "The ticket is about retries; the double charge is an aside at the end. P1 under the SLA."),
    "T-1004": ("P2", None, ["api-conventions", "changelog-2026-03", "errors"], "Retries reusing an Idempotency-Key fail with 400 idempotency_key_reused on live traffic because the retry body differs; asks for the correct retry pattern.", "Needs two pages combined: the conventions rule and the changelog entry that changed it."),
    "T-1005": ("P3", None, ["refunds", "charges-create"], "Second partial refund rejected with amount_exceeds_refundable; asks how to read the remaining refundable balance before refunding.", "Mentions refunds and amounts without any funds being affected; a P3 that sounds like money."),
    "T-1006": ("P1", "money-buried", ["changelog-2026-03", "webhook-signatures-v2", "webhook-signatures-v1"], "SDK 3.0 upgrade made webhook verification fail and was rolled back; two customers were charged twice during the stall; asks the migration order.", "Leads with a failed upgrade; the double charge is a side effect the developer plays down."),
    "T-1007": ("P3", None, ["authentication"], "After a scheduled key rotation the old live key still works two hours later; asks whether that is expected and for how long.", "Sounds like a security incident and is documented behaviour; tests calm reading of docs."),
    "T-1008": ("P3", "no-coverage", [], "Cannot trigger a dispute against a test-mode charge; asks how to simulate one and what the dispute object contains.", "No page covers this. The correct answer is to say so, not to invent a test card."),
    "T-1009": ("P1", "money-buried", ["rate-limits", "api-conventions", "charges-create"], "Bulk import of 1,800 customers times out after about 200 rows; retries have created duplicate invoice charges for a dozen customers.", "Leads with timeouts and batch advice; the duplicate charges are 'somewhat related'."),
    "T-1010": ("P3", "near-miss", ["webhooks-overview"], "Same webhook event delivered several times and out of order; no money affected; asks whether that is expected and how to deduplicate.", "Uses 'duplicate' and 'double' about deliveries, not charges. A keyword rule fires; the SLA does not."),
    "T-1011": ("P1", "money-buried", ["errors", "charges-create", "api-conventions"], "About one in fifty live POST /v1/charges calls returns 502 from eu-west; retries have charged two customers twice.", "Leads with 502s and follows the errors page's own retry advice; the retry is what double-charged customers."),
    "T-1012": ("P3", "no-coverage", [], "Asks whether Apple Pay and Google Pay are supported through Halcyon.js or planned.", "No page covers wallets. Tests whether the assistant admits a documentation gap."),
}

GOLDENS = []
for t in TICKETS:
    severity, trap, pages, summary, reason = GOLDEN_SPEC[t["id"]]
    GOLDENS.append(Golden(name=t["id"], input=ticket_text(t), expected_output=summary,
                          additional_metadata={"expected_severity": severity, "trap": trap,
                                               "pages": pages, "reason": reason}))

CORPUS_FINGERPRINT = corpus_fingerprint(DOCS, TICKETS)
assert CORPUS_FINGERPRINT == "e4485ba0f187", "DOCS or TICKETS drifted from post 0"
print("corpus fingerprint", CORPUS_FINGERPRINT)

corpus fingerprint e4485ba0f187


That is the shared setup cell from the first post in the series, unchanged, and the fingerprint matched. The next cell is the judge and ledger cell from the earlier metric posts. Its body is unchanged. Only the first comment differs, because the one it inherited still said the cell was new. It builds the judge on the same key the assistant uses, passed in explicitly, since DeepEval's `GeminiModel` would otherwise look for `GOOGLE_API_KEY`. A wrapper around the Gemini client records every judge call with its token counts. `score()` runs one metric over a list of cases, a few at a time. It hands back the metric objects so that we can look inside them.

In [2]:
# Carried forward from the earlier metric posts. Everything above is the shared setup cell, unchanged.
import asyncio
import google.genai.models as genai_models
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, SummarizationMetric
from deepeval.models import GeminiModel
from deepeval.models.llms.constants import GEMINI_MODELS_DATA
from deepeval.test_case import LLMTestCase

MONEY = re.compile(r"twice|double|duplicat|second (charge|time)|two (charge|invoice)|charged again|"
                   r"extra charge|multiple charge|overcharg|billed (again|twice)", re.I)   # the trap check
MONEY_TICKET_IDS = ("T-1003", "T-1006", "T-1009", "T-1011")

JUDGE_USAGE = []   # (usage_metadata, temperature sent) for every judge call; DeepEval's counter omits thinking
_generate_content = genai_models.AsyncModels.generate_content

async def counted_generate_content(self, *args, **kwargs):
    response = await _generate_content(self, *args, **kwargs)
    JUDGE_USAGE.append((response.usage_metadata, kwargs["config"].temperature))
    return response

genai_models.AsyncModels.generate_content = counted_generate_content

def judge_calls():
    return len(JUDGE_USAGE)

registry_price = GEMINI_MODELS_DATA[JUDGE_MODEL].input_price, GEMINI_MODELS_DATA[JUDGE_MODEL].output_price
# Same key variable as the app model: GeminiModel would otherwise look for GOOGLE_API_KEY.
judge = GeminiModel(model=JUDGE_MODEL, api_key=os.environ["GEMINI_API_KEY"],
                    cost_per_input_token=PRICE_IN, cost_per_output_token=PRICE_OUT)

async def _measure(metric, case, gate):
    async with gate:
        await metric.a_measure(case, _show_indicator=False)   # no progress spinner in a notebook
    return metric

async def score(metric_cls, cases, **kwargs):
    """One fresh metric per case, MAX_CONCURRENCY cases in flight; returns {case name: measured metric}."""
    gate = asyncio.Semaphore(MAX_CONCURRENCY)
    metrics = [metric_cls(model=judge, **kwargs) for _ in cases]
    await asyncio.gather(*(_measure(m, c, gate) for m, c in zip(metrics, cases)))
    return {c.name: m for c, m in zip(cases, metrics)}

print(f"deepeval {version('deepeval')}   judge {judge.get_model_name()}")
print(f"temperature DeepEval will send: {judge.temperature}   "
      f"(its registry says supports_temperature={judge.supports_temperature()})")
print(f"price per 1M tokens we set: ${PRICE_IN * 1e6:.2f} in, ${PRICE_OUT * 1e6:.2f} out   "
      f"registry default: ${registry_price[0] * 1e6:.2f} in, ${registry_price[1] * 1e6:.2f} out")
print(f"app model calls: {len(USAGE)}   judge calls: {judge_calls()}")

deepeval 4.2.1   judge gemini-3.6-flash (Gemini)
temperature DeepEval will send: 0.0   (its registry says supports_temperature=True)
price per 1M tokens we set: $0.75 in, $3.75 out   registry default: $1.50 in, $7.50 out
app model calls: 0   judge calls: 0


The same four lines the earlier metric posts read. DeepEval 4.2.1 and the judge `gemini-3.6-flash`. The temperature of 0.0 that DeepEval sends whether we like it or not. Our prices beside the registry's, which are next year's. Nothing has been called.

Now the part this post adds. It carries the previous post's retry on a 5xx from Google. Then it makes three decisions and one thing DeepEval does not ship.

The decisions are the retriever's knobs. Embeddings come from `gemini-embedding-001`. An embedding is a list of numbers that stands for a piece of text, placed so that texts with similar meaning sit close together. A chunk is a slice of a page, here whole sentences up to sixty words. Top-k is how many chunks the retriever hands the generator, and it is three. We picked all of this before running anything, which matters later, because one of the traps turns out to depend on it.

The thing DeepEval does not ship is a Gemini embedding model. DeepEval reads embeddings through one interface, `DeepEvalBaseEmbeddingModel`, and only its synthesizer, the tool that writes goldens from documents, ever calls it. So we write the class ourselves. There is one subtlety. Google's embedding model takes a task type. That is a hint saying whether the text is a document waiting to be found or a query looking for one. DeepEval's synthesizer calls `embed_texts` on the chunks it indexes and `embed_text` on the chunk it queries with. So the method name is the only place the right task type can live. Documents go through `embed_texts`. A query goes through `embed_text`. A class with one task type for both would quietly get half its calls wrong, and the synthesizer post would inherit the mistake. Every call passes through `embed()`, which records the texts sent and Google's token count for them. The embedding response itself carries no usage at all.

In [3]:
# New in this post. The two cells above are the setup cell and the judge and ledger cell, unchanged.
# !uv pip install chromadb==1.5.9
import numpy as np
import chromadb
from google import genai
from google.genai import types
from google.genai import errors as genai_errors
from deepeval.metrics import ContextualPrecisionMetric, ContextualRecallMetric, ContextualRelevancyMetric
from deepeval.models import DeepEvalBaseEmbeddingModel
import deepeval.models as deepeval_models

EMBED_MODEL = "gemini-embedding-001"
PRICE_EMBED = 0.15 / 1e6           # USD per input token, standard tier; embeddings have no output tokens
CHUNK_WORDS = 60                   # a chunk holds whole sentences, up to this many words
TOP_K = 3                          # chunks the retriever hands to the generator

# The previous post's change to the ledger wrapper: judge calls retry on a 5xx from Google, and each
# failed attempt is recorded so the ledger can report it.
JUDGE_RETRIES = []

async def retrying_generate_content(self, *args, **kwargs):
    for attempt in range(5):
        try:
            response = await _generate_content(self, *args, **kwargs)   # the original client method, saved above
            break
        except genai_errors.ServerError as exc:
            JUDGE_RETRIES.append(type(exc).__name__)
            if attempt == 4:
                raise
            await asyncio.sleep(2.0 * 2 ** attempt)
    JUDGE_USAGE.append((response.usage_metadata, kwargs["config"].temperature))
    return response

genai_models.AsyncModels.generate_content = retrying_generate_content

# Every embedding call goes through embed(), so the ledger sees all of them.
EMBED_CLIENT = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
EMBED_USAGE = []   # one dict per embed_content call: texts sent and Google's token count for them

def embed(texts, task_type):
    """One embed_content call. Returns one vector per text, in the order the texts were given."""
    config = types.EmbedContentConfig(task_type=task_type)
    response = EMBED_CLIENT.models.embed_content(model=EMBED_MODEL, contents=texts, config=config)
    counted = EMBED_CLIENT.models.count_tokens(model=EMBED_MODEL, contents=texts)   # embed responses carry no usage
    EMBED_USAGE.append({"texts": len(texts), "tokens": counted.total_tokens})
    vectors = []
    for embedding in response.embeddings:
        vectors.append(embedding.values)
    return vectors

class GeminiEmbedder(DeepEvalBaseEmbeddingModel):
    """gemini-embedding-001 behind DeepEval's embedding interface. embed_texts is for documents, embed_text for a query."""
    def __init__(self):
        super().__init__(EMBED_MODEL)

    def load_model(self):
        return EMBED_CLIENT

    def embed_texts(self, texts):
        return with_retry(embed, texts, "RETRIEVAL_DOCUMENT")

    def embed_text(self, text):
        return with_retry(embed, [text], "RETRIEVAL_QUERY")[0]

    async def a_embed_texts(self, texts):
        return self.embed_texts(texts)

    async def a_embed_text(self, text):
        return self.embed_text(text)

    def get_model_name(self):
        return EMBED_MODEL

embedder = GeminiEmbedder()

shipped = []
for name in dir(deepeval_models):
    if name.endswith("EmbeddingModel") and name != "DeepEvalBaseEmbeddingModel":
        shipped.append(name)
print(f"embedding classes deepeval {version('deepeval')} ships: {', '.join(shipped)}")
print(f"ours: {embedder.get_model_name()} inside {type(embedder).__name__}, a subclass of DeepEvalBaseEmbeddingModel")
print(f"judge wrapper now: {genai_models.AsyncModels.generate_content.__name__}, up to 5 attempts per call")
print(f"chunk budget {CHUNK_WORDS} words, top-k {TOP_K}   app model calls: {len(USAGE)}   judge calls: {judge_calls()}   embed calls: {len(EMBED_USAGE)}")


embedding classes deepeval 4.2.1 ships: AzureOpenAIEmbeddingModel, LocalEmbeddingModel, OllamaEmbeddingModel, OpenAIEmbeddingModel
ours: gemini-embedding-001 inside GeminiEmbedder, a subclass of DeepEvalBaseEmbeddingModel
judge wrapper now: retrying_generate_content, up to 5 attempts per call
chunk budget 60 words, top-k 3   app model calls: 0   judge calls: 0   embed calls: 0


Four embedding classes in the box, and none of them is Gemini. Ours is now beside them, a subclass of the same base. Still nothing called.

## A retriever worth scoring, and the chunk that splits the trap

A retriever needs an index. The chunker packs whole sentences into chunks of at most sixty words, never splitting a sentence, then embeds them all in one call. Each chunk keeps its embedding, the vector, on the record next to the page's `slug`, `status` and `updated` fields. The generator will never see those three, and that is the whole story.

Then the check the whole post depends on. The deprecated signature page announces its own deprecation in its final sentence. Hand a model the whole page and it will read that sentence and decline the old scheme. The trap only works if chunking puts the verification instructions and the deprecation notice in different chunks. So before building anything on it, we print the chunk that holds the SHA1 instructions and assert that the word deprecated is not in it.

In [4]:
def sentences(text):
    return [s for s in re.split(r"(?<=[.?!])\s+", text) if s]

def chunk_page(doc):
    """Whole sentences packed into chunks of at most CHUNK_WORDS words. A sentence is never split."""
    chunks = []
    current = []
    for sentence in sentences(doc["body"]):
        words_so_far = len(" ".join(current).split())
        if current and words_so_far + len(sentence.split()) > CHUNK_WORDS:
            chunks.append(" ".join(current))
            current = []
        current.append(sentence)
    if current:
        chunks.append(" ".join(current))
    return chunks

CHUNKS = []   # one record per chunk: part, slug, status, updated, text, and after indexing, vector
for doc in DOCS:
    parts = chunk_page(doc)
    for number, text in enumerate(parts, 1):
        CHUNKS.append({"part": f"{doc['slug']}#{number}", "slug": doc["slug"], "status": doc["status"],
                       "updated": doc["updated"], "text": text})

texts = []
for chunk in CHUNKS:
    texts.append(chunk["text"])
t0 = time.time()
vectors = embedder.embed_texts(texts)
for chunk, vector in zip(CHUNKS, vectors):
    chunk["vector"] = vector
print(f"{len(CHUNKS)} chunks from {len(DOCS)} pages, embedded in {len(EMBED_USAGE)} call, "
      f"{len(vectors[0])} dimensions each, {time.time() - t0:.1f} s\n")

print(f"{'chunk':<26}{'status':<12}{'words':>5}")
for chunk in CHUNKS:
    print(f"{chunk['part']:<26}{chunk['status']:<12}{len(chunk['text'].split()):>5}")

# The trap check. The chunk that carries the SHA1 instructions must not carry the deprecation notice.
sha1_chunk = None
for chunk in CHUNKS:
    if chunk["slug"] == "webhook-signatures-v1" and "HMAC-SHA1" in chunk["text"]:
        sha1_chunk = chunk
deprecation_in_sha1_chunk = "deprecated" in sha1_chunk["text"]
print(f"\n{sha1_chunk['part']}, status {sha1_chunk['status']}, updated {sha1_chunk['updated']}:")
print(sha1_chunk["text"])
print(f"\ndeprecation notice inside this chunk: {deprecation_in_sha1_chunk}")
assert not deprecation_in_sha1_chunk, "the chunker did not separate the instructions from the deprecation notice"


30 chunks from 12 pages, embedded in 1 call, 3072 dimensions each, 0.8 s

chunk                     status      words
quickstart#1              current        34
quickstart#2              current        55
quickstart#3              current        19
api-conventions#1         current        43
api-conventions#2         current        42
api-conventions#3         current        29
authentication#1          current        35
authentication#2          current        50
authentication#3          current        15
errors#1                  current        49
errors#2                  current        29
rate-limits#1             current        42
rate-limits#2             current        51
charges-create#1          current        56
charges-create#2          current        39
refunds#1                 current        56
refunds#2                 current        45
webhooks-overview#1       current        27
webhooks-overview#2       current        46
webhooks-overview#3       current        40
we

It split. Thirty chunks from twelve pages, each between 15 and 56 words, every page cut into two or three pieces. The deprecated page became two. The first, `webhook-signatures-v1#1`, is 53 words of perfectly good instructions. The header name, the algorithm, the secret, the constant-time comparison, the rejection rule. The sentence that says all of it is obsolete lives in `webhook-signatures-v1#2`. The assertion passed.

Read as the generator will read it, that chunk has nothing in it to say the page is dead. The record knows. Its `status` field says deprecated and its `updated` field says when. In a real docs store that is where version currency lives, in metadata beside the text, and chunking keeps the text. What the generator reads is the text.

The corpus pages and tickets from the first post do not change. What this post adds is a question set beside them. Ten developer questions, each with the pages that answer it and a reference answer written from those pages. Eight are ordinary. Two carry a trap. The webhook question can only be answered correctly from the current signature page. The charge question is answered by the charge page, except for one rule. Every money-moving request must carry an `Idempotency-Key` header. That rule sits on the conventions page, which nobody asking how to create a charge would think to name. We re-print the corpus fingerprint to show the canonical data is untouched and fingerprint the questions on their own.

In [5]:
# Ten developer questions with reference answers. The pages field names where the answer lives in DOCS.
DOC_QUESTIONS = [
    {"id": "Q-01", "trap": "deprecated-page", "pages": ["webhook-signatures-v2"],
     "question": "How do I verify the signature on a webhook Halcyon sends me?",
     "expected": "Read the Halcyon-Signature header, which has the form t=<unix timestamp>,v2=<hex digest>. Compute HMAC-SHA256 over the timestamp, a period and the raw request body with the endpoint's signing secret (whsec_), compare it with the v2 value using a constant-time comparison, and reject the event if the timestamp is more than 300 seconds old."},
    {"id": "Q-02", "trap": "missing-requirement", "pages": ["charges-create", "api-conventions"],
     "question": "How do I create a charge?",
     "expected": "POST /v1/charges with amount in minor units, a lowercase ISO 4217 currency and a source token from Halcyon.js or a saved payment method id starting pm_. Because charges move money, include an Idempotency-Key header with a client-generated unique string such as a UUID v4, so that a retried request returns the original response instead of creating a second charge."},
    {"id": "Q-03", "trap": None, "pages": ["rate-limits"],
     "question": "What is the rate limit in test mode?",
     "expected": "Test mode allows 25 requests per second. Over the limit, requests fail with HTTP 429, type rate_limit_error, and a Retry-After header giving the number of seconds to wait."},
    {"id": "Q-04", "trap": None, "pages": ["authentication"],
     "question": "After I rotate my secret key, how long does the old key keep working?",
     "expected": "The new key is active immediately and the old key keeps working for 24 hours so deployments can roll over. It can be revoked sooner from the dashboard if it was exposed."},
    {"id": "Q-05", "trap": None, "pages": ["refunds"],
     "question": "Can I refund part of a charge, and can I do it more than once?",
     "expected": "Yes. POST /v1/refunds with the charge id and an amount refunds part of it. A charge can be refunded several times until the refunded total reaches the original amount. A request beyond the remaining balance returns 400 with code amount_exceeds_refundable."},
    {"id": "Q-06", "trap": None, "pages": ["errors"],
     "question": "Should I retry a request that came back with a 5xx error?",
     "expected": "Yes. A request that returned 5xx may have partially completed, so retry it with exponential backoff and quote the request_id if you contact support. Do not retry 4xx errors other than 429."},
    {"id": "Q-07", "trap": None, "pages": ["quickstart"],
     "question": "Which test card numbers give me a successful payment and a decline?",
     "expected": "Use 4242 4242 4242 4242 for a successful payment and 4000 0000 0000 0002 for a decline."},
    {"id": "Q-08", "trap": None, "pages": ["webhooks-overview"],
     "question": "If my webhook endpoint is down, how long will Halcyon keep retrying deliveries?",
     "expected": "Deliveries are retried with exponential backoff, at 1 minute, 5 minutes, 30 minutes, 2 hours and then every 6 hours, for up to 72 hours. After that the event is marked failed and shown in the dashboard."},
    {"id": "Q-09", "trap": None, "pages": ["charges-create"],
     "question": "How do I authorize a card now and capture the payment later?",
     "expected": "Pass capture=false when creating the charge to authorize only, then capture within 7 days with POST /v1/charges/{id}/capture."},
    {"id": "Q-10", "trap": None, "pages": ["changelog-2026-03", "webhook-signatures-v2"],
     "question": "What changed for webhook signatures in API version 2026-03-01?",
     "expected": "Webhook signatures v2, HMAC-SHA256 with a timestamped Halcyon-Signature header, became the default. v1, HMAC-SHA1 with the X-Halcyon-Signature header, is deprecated and is sent only to accounts pinned to earlier versions. SDK 3.0 verifies v2 only and SDK 2.x verifies v1 only."},
]
QUESTION_BY_ID = {}
for question in DOC_QUESTIONS:
    QUESTION_BY_ID[question["id"]] = question

def question_fingerprint(questions):
    return hashlib.sha256(json.dumps(questions, sort_keys=True).encode()).hexdigest()[:12]

assert corpus_fingerprint(DOCS, TICKETS) == "e4485ba0f187", "DOCS or TICKETS drifted from post 0"
print(f"corpus fingerprint    {corpus_fingerprint(DOCS, TICKETS)}   the canonical pages and tickets, untouched")
print(f"question fingerprint  {question_fingerprint(DOC_QUESTIONS)}   {len(DOC_QUESTIONS)} questions, new in this post\n")
print(f"{'id':<6}{'trap':<21}{'pages that answer it':<42}question")
for question in DOC_QUESTIONS:
    trap = question["trap"] or ""
    print(f"{question['id']:<6}{trap:<21}{', '.join(question['pages']):<42}{question['question']}")


corpus fingerprint    e4485ba0f187   the canonical pages and tickets, untouched
question fingerprint  68e3908a66a7   10 questions, new in this post

id    trap                 pages that answer it                      question
Q-01  deprecated-page      webhook-signatures-v2                     How do I verify the signature on a webhook Halcyon sends me?
Q-02  missing-requirement  charges-create, api-conventions           How do I create a charge?
Q-03                       rate-limits                               What is the rate limit in test mode?
Q-04                       authentication                            After I rotate my secret key, how long does the old key keep working?
Q-05                       refunds                                   Can I refund part of a charge, and can I do it more than once?
Q-06                       errors                                    Should I retry a request that came back with a 5xx error?
Q-07                       quickstart       

The corpus fingerprint is `e4485ba0f187`, the value the first post shipped. The question set gets its own, `68e3908a66a7`, and later posts assert it the same way. Q-01 is the webhook question. The current signature page answers it, and a deprecated page that reads like the right answer sits next to it. Q-02 names one page and needs two. The other eight are controls, one page each, except Q-10, which needs the changelog and the current signature page.

Now the assistant. It is a graph with two nodes. `retrieve` embeds the question, scores every chunk by cosine similarity, and keeps the top three. Cosine similarity is a number between minus one and one that says how alike two vectors point, and numpy computes it in one line. It writes two things into the state. `retrieved` holds the chunk records, status and date included. `context` holds the chunk texts alone, and the text is all the generator sees. `generate` is `respond()` from the setup cell. It is the same node that summarized tickets in the earlier posts, now handed pages instead. The retriever goes in as a function. A later cell can swap it for a broken one, or for a real database, without touching the generator.

`inline_retriever()` takes one more argument, `highest_first`. Set it to false and the sort runs the other way, returning the three chunks least like the question. That is exactly what happens when code written for a similarity is handed a distance, where smaller is better. We will need it soon.

We run all ten questions and look at the webhook question in full. Which chunks came back, with their status. Then what the generator made of them.

In [6]:
class DocsState(TypedDict):
    question: str
    retrieved: list[dict]        # chunk records: part, slug, status, updated, text, vector
    context: list[str] | None    # the chunk texts, which is all the generator sees
    answer: str

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def inline_retriever(top_k, highest_first=True):
    """A retriever over CHUNKS. highest_first=False reproduces a distance-for-similarity bug: the k chunks least like the question."""
    def retrieve_chunks(question_text):
        query_vector = embedder.embed_text(question_text)
        similarity = {}                                    # similarity[chunk part] -> cosine to the question
        for chunk in CHUNKS:
            similarity[chunk["part"]] = cosine(query_vector, chunk["vector"])
        ranked = sorted(CHUNKS, key=lambda chunk: similarity[chunk["part"]], reverse=highest_first)
        return ranked[:top_k]
    return retrieve_chunks

def build_assistant(retrieve_chunks):
    """Two nodes. The retriever passed in, then the setup cell's respond() as the generator."""
    def retrieve(state: DocsState) -> dict:
        retrieved = retrieve_chunks(state["question"])
        context = []
        for chunk in retrieved:
            context.append(chunk["text"])                  # status and updated stay on the record, not in the text
        return {"retrieved": retrieved, "context": context}

    builder = StateGraph(DocsState)
    builder.add_node("retrieve", retrieve)
    builder.add_node("generate", respond)
    builder.add_edge(START, "retrieve")
    builder.add_edge("retrieve", "generate")
    builder.add_edge("generate", END)
    return builder.compile()

def ask(assistant, question):
    """One question through one assistant. The result keeps the question record, the chunks retrieved and the answer."""
    state = assistant.invoke({"question": question["question"]})
    return {"id": question["id"], "question": question, "retrieved": state["retrieved"],
            "context": state["context"], "answer": state["answer"]}

def ask_all(assistant, questions):
    """Every question through one assistant, MAX_CONCURRENCY at a time, as post 0 ran the tickets. Keyed by question id."""
    results = {}
    with ThreadPoolExecutor(MAX_CONCURRENCY) as pool:
        futures = []
        for question in questions:
            futures.append(pool.submit(ask, assistant, question))
        for future in futures:
            result = future.result()
            results[result["id"]] = result
    return results

V1_SCHEME = re.compile(r"HMAC-SHA1\b|X-Halcyon-Signature", re.I)
V2_SCHEME = re.compile(r"HMAC-SHA256|300 seconds", re.I)

ASSISTANT = build_assistant(inline_retriever(TOP_K))
t0 = time.time()
RESULTS = {"healthy": ask_all(ASSISTANT, DOC_QUESTIONS)}   # RESULTS[config][question id] -> result
print(f"{len(RESULTS['healthy'])} questions answered in {time.time() - t0:.0f} s\n")

webhook = RESULTS["healthy"]["Q-01"]
print(webhook["question"]["question"])
print("-" * 78)
for rank, chunk in enumerate(webhook["retrieved"], 1):
    print(f"rank {rank}  {chunk['part']:<26}status {chunk['status']:<11}updated {chunk['updated']}")
print("-" * 78)
print(webhook["answer"])
print("-" * 78)
names_v1 = bool(V1_SCHEME.search(webhook["answer"]))
names_v2 = bool(V2_SCHEME.search(webhook["answer"]))
print(f"answer names the v1 scheme, HMAC-SHA1 or X-Halcyon-Signature: {names_v1}")
print(f"answer names the v2 scheme, HMAC-SHA256 or the 300 second rule: {names_v2}")
print(f"app model calls: {len(USAGE)}   embed calls: {len(EMBED_USAGE)}   judge calls: {judge_calls()}")


10 questions answered in 11 s

How do I verify the signature on a webhook Halcyon sends me?
------------------------------------------------------------------------------
rank 1  webhook-signatures-v1#1   status deprecated updated 2026-03-01
rank 2  webhook-signatures-v2#1   status current    updated 2026-03-01
rank 3  webhook-signatures-v1#2   status deprecated updated 2026-03-01
------------------------------------------------------------------------------
Based on the provided documentation, signature verification depends on which scheme/API version you are using:

### For `webhook-signatures-v2` (API versions on or after 2026-03-01)
1. Parse the `Halcyon-Signature` header, which is formatted as `t=<unix timestamp>,v2=<hex digest>`.
2. Construct the signed payload by concatenating the timestamp, a period, and the raw request body (`<timestamp>.<raw_body>`).
3. Compute the HMAC-SHA256 digest of that signed payload using your secret.
4. Compare your computed digest with the `v2` diges

The deprecated page won. Rank 1 is `webhook-signatures-v1#1`, the SHA1 instructions. Rank 2 is the first chunk of the current page. Rank 3 is the other half of the deprecated page, the chunk with the notice in it. The generator read all three and wrote a careful answer. The v2 scheme first, then the SHA1 scheme marked as legacy, for API versions before March. Both flags say True. The answer names both schemes.

So on this run the trap half fired. The deprecated page won the ranking, and then its own last sentence, arriving at rank 3, told the generator not to trust it. That rescue is an accident of a corpus with thirty chunks, where both halves of a short page fit in the top three. A later section takes it away. What matters now is the ranking. That is what the retriever metrics are about to grade.

## The retriever metrics: precision, recall, relevancy

The previous post scored Contextual Recall and Contextual Relevancy against a fixture, a stand-in for a retriever. The triage assistant had no retriever, so we handed them the ticket as the retrieved text and they graded the ticket against itself. That debt comes due here. The retrieved text is now three chunks a real retriever chose. The reference answer was written from a page it did not rank first.

One test case from the webhook run. `input` is the question, `actual_output` the answer, `expected_output` the reference, `retrieval_context` the three chunk texts in rank order. We print which fields each metric declares it reads. Then we score the case with all five, one metric at a time, and open every score. Faithfulness has sat on the reserved list since the metric-library post, waiting for a retriever. This is what it was waiting for.

Then a repeat of the previous post's probe, with the third member of the trio added. Replace the answer with the words "(no answer)" and score the three retriever metrics again.

In [7]:
def test_case(result):
    """A DeepEval test case from one result: the question, the answer, the reference answer and the chunk texts."""
    return LLMTestCase(name=result["id"], input=result["question"]["question"], actual_output=result["answer"],
                       expected_output=result["question"]["expected"], retrieval_context=result["context"])

METRICS = {
    "Contextual Precision": ContextualPrecisionMetric,
    "Contextual Recall": ContextualRecallMetric,
    "Contextual Relevancy": ContextualRelevancyMetric,
    "Faithfulness": FaithfulnessMetric,
    "Answer Relevancy": AnswerRelevancyMetric,
}
RETRIEVER_METRICS = ["Contextual Precision", "Contextual Recall", "Contextual Relevancy"]

print(f"{'metric':<22}fields it declares it reads")
for name, metric_class in METRICS.items():
    fields = []
    for param in metric_class._required_params:
        fields.append(param.value)
    print(f"{name:<22}{', '.join(fields)}")

case = test_case(webhook)
before = judge_calls()
SINGLE = {}     # SINGLE[metric name] -> the measured metric object for this one case
for name, metric_class in METRICS.items():
    metric = metric_class(model=judge, threshold=None)
    await metric.a_measure(case, _show_indicator=False)
    SINGLE[name] = metric

precision = SINGLE["Contextual Precision"]
print(f"\nContextual Precision {precision.score:.2f}   one verdict per chunk, in rank order")
for rank, verdict in enumerate(precision.verdicts, 1):
    part = webhook["retrieved"][rank - 1]["part"]
    print(f"  rank {rank} {part:<26}{verdict.verdict:<4} {verdict.reason}")

recall = SINGLE["Contextual Recall"]
print(f"\nContextual Recall {recall.score:.2f}   one verdict per sentence of the reference answer")
for verdict in recall.verdicts:
    print(f"  {verdict.verdict:<4} {verdict.reason}")

relevancy = SINGLE["Contextual Relevancy"]
print(f"\nContextual Relevancy {relevancy.score:.2f}   statements the judge cut from each chunk, and how many it marked relevant")
for rank, chunk_verdicts in enumerate(relevancy.verdicts_list, 1):
    relevant = 0
    for verdict in chunk_verdicts.verdicts:
        if verdict.verdict == "yes":
            relevant += 1
    part = webhook["retrieved"][rank - 1]["part"]
    print(f"  rank {rank} {part:<26}{relevant} of {len(chunk_verdicts.verdicts)} relevant")

faithfulness = SINGLE["Faithfulness"]
contradicted = 0
for verdict in faithfulness.verdicts:
    if verdict.verdict == "no":
        contradicted += 1
print(f"\nFaithfulness {faithfulness.score:.2f}   {len(faithfulness.claims)} claims in the answer, "
      f"{len(faithfulness.truths)} truths in the chunks, {contradicted} claims contradicted")
print(f"  reason: {faithfulness.reason}")

answer_relevancy = SINGLE["Answer Relevancy"]
print(f"\nAnswer Relevancy {answer_relevancy.score:.2f}   {len(answer_relevancy.statements)} statements in the answer")
print(f"  reason: {answer_relevancy.reason}")
print(f"\njudge calls for this one case, five metrics: {judge_calls() - before}")

# The previous post showed two of the trio scoring the same with the summary replaced. Now the trio has its third member.
blank = LLMTestCase(name="Q-01", input=case.input, actual_output="(no answer)",
                    expected_output=case.expected_output, retrieval_context=case.retrieval_context)
before = judge_calls()
print("\nthe same case with actual_output replaced by '(no answer)':")
for name in RETRIEVER_METRICS:
    metric = METRICS[name](model=judge, threshold=None)
    await metric.a_measure(blank, _show_indicator=False)
    print(f"  {name:<22}{metric.score:.2f}   with the real answer: {SINGLE[name].score:.2f}")
print(f"judge calls: {judge_calls() - before}")


metric                fields it declares it reads
Contextual Precision  input, retrieval_context, expected_output
Contextual Recall     input, retrieval_context, expected_output
Contextual Relevancy  input, retrieval_context
Faithfulness          input, actual_output, retrieval_context
Answer Relevancy      input, actual_output



Contextual Precision 1.00   one verdict per chunk, in rank order
  rank 1 webhook-signatures-v1#1   yes  It provides details on using the endpoint's signing secret (whsec_) and comparing values using a constant-time comparison.
  rank 2 webhook-signatures-v2#1   yes  It specifies the Halcyon-Signature header format 't=<unix timestamp>,v2=<hex digest>' and that the signed payload consists of the timestamp, a period, and the raw request body.
  rank 3 webhook-signatures-v1#2   yes  It explains that the current signature scheme uses HMAC-SHA256 with a timestamped Halcyon-Signature header.

Contextual Recall 0.50   one verdict per sentence of the reference answer
  yes  The 2nd node directly states that 'Each webhook request carries a Halcyon-Signature header of the form t=<unix timestamp>,v2=<hex digest>.'
  no   While the 1st, 2nd, and 3rd nodes support computing HMAC-SHA256 with the signing secret (whsec_) and using constant-time comparison, none of the nodes state that the event shoul

  Contextual Precision  1.00   with the real answer: 1.00


  Contextual Recall     0.50   with the real answer: 0.50


  Contextual Relevancy  1.00   with the real answer: 1.00
judge calls: 8


The declarations first, because they are the split written down. The three retriever metrics declare `retrieval_context` and not one of them declares `actual_output`. The two generator metrics declare `actual_output`, and only Faithfulness also reads the chunks.

Then the scores, and the first surprise. Contextual Precision 1.00. Three chunks, three yes verdicts. The judge looked at the deprecated chunk at rank 1 and marked it useful. It mentions the `whsec_` secret and the constant-time comparison, and both phrases appear in the reference answer. The deprecated page ranked first, and the metric that grades ranking gave it full marks. The judge matched two phrases against the reference and never weighed the algorithm.

Contextual Recall 0.50. The reference has two sentences. The first, the header format, was found in the second chunk. The second was not, and the judge says why. Nothing in the chunks mentions rejecting an event whose timestamp is more than 300 seconds old. That rule sits in the second chunk of the current page, which was never retrieved. Recall saw the gap. Precision could not, because precision only grades what arrived.

Contextual Relevancy 1.00. Eight statements across the three chunks, every one marked relevant. The SHA1 instructions are about verifying a webhook signature. So is the deprecation notice. Relevancy asks whether the chunks are about the question, and they are.

Faithfulness 1.00. The judge cut the answer into nine claims and the chunks into six truths, then asked one question of each claim. Does any chunk contradict it. None did, so nine of nine count as faithful, and that is the whole test. A claim the chunks neither support nor contradict gets an idk verdict, and idk counts as faithful unless `penalize_ambiguous_claims` is switched on. Answer Relevancy 1.00, twelve statements, every one about the question. Both metrics did their job. The generator did not contradict the pages it was given, and it answered the question. Neither metric reads a page's status, and one of the pages was dead.

The arithmetic behind those five numbers is short. Precision averages, over the useful chunks only, the fraction of useful chunks at each rank. Three yes verdicts give 1 of 1, 2 of 2 and 3 of 3, and the average is 1.00. Recall is yes sentences over reference sentences, 1 of 2. Relevancy pools the statements from every chunk, 3 plus 2 plus 3 relevant out of the same 8, so 1.00. Faithfulness counts every claim that is not contradicted, 9 of 9. Answer Relevancy counts every statement not marked off-topic, 12 of 12. Only precision has any weighting, and it weights order, never truth.

Then the placeholder. With the answer replaced by "(no answer)", precision scored 1.00, recall 0.50 and relevancy 1.00. Identical. The previous post showed this for two of the trio. Now it holds for all three. The trio grades the retriever and never opens the answer.

### ContextualPrecisionMetric
**Plain meaning:** Did the retriever put the useful chunks ahead of the useless ones?
**Under the hood:** The judge reads each chunk in `retrieval_context` in rank order. It decides whether each was useful for arriving at `expected_output`, one yes or no per chunk. The score is a weighted cumulative precision. At each rank that holds a useful chunk, it takes the fraction of useful chunks so far, then averages those fractions over the useful chunks. A useless chunk ranked above a useful one lowers the score. A useless chunk ranked last does not. Two judge calls per test case, the verdicts and then the reason, whatever the number of chunks.
**Needs:** `input`, `expected_output` and `retrieval_context`. Reference-based. The docs list `actual_output` as required too. The package does not declare it, and the probe above shows the score does not change when the answer is replaced.
**Confused with:** `ContextualRecallMetric`, carded in the previous post, which reads the same three fields. Recall asks whether each sentence of the reference can be found anywhere in the retrieved chunks. Precision asks whether the useful chunks are ranked above the useless ones. The tell: recall does not care about order, and precision does not care whether anything is missing. Three chunks that all help score 1.00 on precision even when the chunk with the answer was never retrieved.
**Use when:** the question is whether the order the generator reads chunks in is right. That order comes from a similarity score or a reranker, a second model that reorders retrieved chunks.
**Do not use when:** the question is whether something is missing. That is recall. Nor when there is no reference answer, because the judge decides usefulness against `expected_output` and the metric raises without it.

**Precision against recall, in one line.** Precision is about order and cannot see absence. Recall is about presence and cannot see order. The webhook case scored 1.00 on precision and 0.50 on recall. Three useful chunks in a fine order, and the one with the algorithm left behind. Both numbers are right. They answer different questions.

The other three metrics were carded earlier. Contextual Recall and Contextual Relevancy in the previous post, Faithfulness and Answer Relevancy in the first metric post. One note carries forward. The docs, as we write this, list `actual_output` as required for both Contextual Precision and Contextual Recall. The package declares it for neither, and the placeholder probe moved none of the three scores.

## Breaking retrieval on purpose, and reading the signature twice

A metric is only worth trusting once we have seen it move. So we break the retriever and keep everything else. The break is the distance-for-similarity bug from `inline_retriever()`. Vector databases return distances, where smaller means closer, and code written for a similarity sorts the other way. The result is a retriever that confidently returns the three chunks least like every question. Same generator, same prompt, same questions.

Before any judge reads anything, we want to see the damage with our own eyes. Which chunks came back, whether the golden's pages are among them, and whether the answer admits the pages do not cover the question. That last column is a string check, not a judgement.

In [8]:
BROKEN = build_assistant(inline_retriever(TOP_K, highest_first=False))
t0 = time.time()
RESULTS["broken"] = ask_all(BROKEN, DOC_QUESTIONS)
print(f"{len(RESULTS['broken'])} questions answered by the broken configuration in {time.time() - t0:.0f} s\n")

CASES = {}      # CASES[config] -> test cases in question order
for config in RESULTS:
    CASES[config] = []
    for question in DOC_QUESTIONS:
        CASES[config].append(test_case(RESULTS[config][question["id"]]))

NOT_COVERED = re.compile(r"no (information|mention)|not covered|(does|do) not (cover|contain|address|mention|specify|state|provide|include)", re.I)   # a string check, not a judgement

def pages_hit(result):
    """How many of the pages named in the golden have at least one chunk in the retrieved set, out of how many."""
    retrieved_slugs = set()
    for chunk in result["retrieved"]:
        retrieved_slugs.add(chunk["slug"])
    hit = 0
    for slug in result["question"]["pages"]:
        if slug in retrieved_slugs:
            hit += 1
    return f"{hit} of {len(result['question']['pages'])}"

for question in DOC_QUESTIONS:
    print(question["id"], question["question"])
    for config in RESULTS:
        result = RESULTS[config][question["id"]]
        parts = ", ".join(chunk["part"] for chunk in result["retrieved"])
        says_not_covered = "yes" if NOT_COVERED.search(result["answer"]) else "no"
        print(f"    {config:<9}pages hit {pages_hit(result):<7}says not covered {says_not_covered:<4} {parts}")
print(f"\napp model calls: {len(USAGE)}   embed calls: {len(EMBED_USAGE)}   judge calls: {judge_calls()}")


10 questions answered by the broken configuration in 7 s

Q-01 How do I verify the signature on a webhook Halcyon sends me?
    healthy  pages hit 1 of 1 says not covered no   webhook-signatures-v1#1, webhook-signatures-v2#1, webhook-signatures-v1#2
    broken   pages hit 0 of 1 says not covered yes  support-sla#2, rate-limits#1, refunds#1
Q-02 How do I create a charge?
    healthy  pages hit 1 of 2 says not covered no   quickstart#2, charges-create#1, charges-create#2
    broken   pages hit 0 of 2 says not covered yes  authentication#2, webhook-signatures-v2#3, webhook-signatures-v1#2
Q-03 What is the rate limit in test mode?
    healthy  pages hit 1 of 1 says not covered no   rate-limits#1, rate-limits#2, quickstart#3
    broken   pages hit 0 of 1 says not covered yes  webhook-signatures-v1#2, support-sla#2, webhook-signatures-v1#1
Q-04 After I rotate my secret key, how long does the old key keep working?
    healthy  pages hit 1 of 1 says not covered no   authentication#2, webhook-s

The damage is total, and it took seven seconds. The healthy configuration hit at least one chunk of every expected page on every question but one. On the charge question it hit one page of two, which is the second trap showing early. The broken configuration hit nothing, on any question, and every one of its answers says the pages do not cover the question. No healthy answer does. A string check already knows which configuration is broken. The question is whether the metrics do, and by how much.

So, the panel. `score_panel()` runs each metric over the ten cases of each configuration, four cases in flight at a time. We get every score, healthy beside broken, then the mean per metric and one more number. On how many of the ten questions did the broken configuration score strictly below the healthy one. That count is the separation, and the bar is not a lower average. A metric that catches a broken retriever should separate on every question.

In [9]:
async def score_panel(cases_by_config):
    """Every metric over every case of every configuration. panel[config][metric][question id] -> measured metric."""
    panel = {}
    for config, cases in cases_by_config.items():
        panel[config] = {}
        for name, metric_class in METRICS.items():
            before = judge_calls()
            panel[config][name] = await score(metric_class, cases, threshold=None)
            CALLS_PER_CASE[name] = (judge_calls() - before) // len(cases)
    return panel

def mean_score(panel, config, name):
    scores = []
    for question in DOC_QUESTIONS:
        scores.append(panel[config][name][question["id"]].score)
    return statistics.mean(scores)

def broken_below_healthy(panel, name):
    """How many questions the broken configuration scored strictly below the healthy one on this metric."""
    count = 0
    for question in DOC_QUESTIONS:
        healthy_score = panel["healthy"][name][question["id"]].score
        broken_score = panel["broken"][name][question["id"]].score
        if broken_score < healthy_score:
            count += 1
    return count

PANEL = {}            # PANEL[run][config][metric][question id] -> the measured metric object
CALLS_PER_CASE = {}   # judge calls one metric spends on one case, from the last configuration scored
RUN_CALLS = {}
panel_start = judge_calls()
t0 = time.time()
PANEL[1] = await score_panel(CASES)
RUN_CALLS[1] = judge_calls() - panel_start
print(f"run 1: {RUN_CALLS[1]} judge calls, {time.time() - t0:.0f} s\n")

print(f"{'id':<7}{'precision':<13}{'recall':<13}{'relevancy':<13}{'faithful':<13}{'answer rel':<13}   healthy  broken")
for question in DOC_QUESTIONS:
    row = f"{question['id']:<7}"
    for name in METRICS:
        healthy_score = PANEL[1]["healthy"][name][question["id"]].score
        broken_score = PANEL[1]["broken"][name][question["id"]].score
        row += f"{healthy_score:>5.2f}  {broken_score:>5.2f}  "
    print(row)

print(f"\n{'metric':<22}{'healthy mean':>13}{'broken mean':>13}   broken below healthy on")
for name in METRICS:
    healthy_mean = mean_score(PANEL[1], "healthy", name)
    broken_mean = mean_score(PANEL[1], "broken", name)
    print(f"{name:<22}{healthy_mean:>13.2f}{broken_mean:>13.2f}   {broken_below_healthy(PANEL[1], name)} of {len(DOC_QUESTIONS)} questions")


run 1: 300 judge calls, 284 s

id     precision    recall       relevancy    faithful     answer rel      healthy  broken
Q-01    1.00   0.00   0.50   0.00   1.00   0.00   1.00   1.00   1.00   1.00  
Q-02    1.00   0.00   0.50   0.00   0.71   0.00   1.00   1.00   1.00   1.00  
Q-03    1.00   0.00   1.00   0.00   0.25   0.00   1.00   1.00   1.00   1.00  
Q-04    1.00   0.00   1.00   0.00   0.43   0.00   1.00   1.00   1.00   1.00  
Q-05    1.00   0.00   0.75   0.00   0.44   0.00   1.00   1.00   1.00   1.00  
Q-06    1.00   0.00   1.00   0.00   0.56   0.00   1.00   1.00   0.75   1.00  
Q-07    1.00   0.00   1.00   0.00   0.22   0.00   1.00   1.00   1.00   1.00  
Q-08    1.00   0.00   1.00   0.00   0.38   0.00   1.00   1.00   1.00   1.00  
Q-09    1.00   0.00   1.00   0.00   0.25   0.00   1.00   1.00   1.00   1.00  
Q-10    1.00   0.00   1.00   0.00   0.88   0.00   1.00   1.00   1.00   1.00  

metric                 healthy mean  broken mean   broken below healthy on
Contextual Precision  

Three hundred judge calls, 284 seconds, and the table reads in column pairs, healthy on the left, broken on the right.

The retriever trio did what a smoke alarm should. Precision, 1.00 on every healthy question and 0.00 on every broken one. Recall, between 0.50 and 1.00 healthy and 0.00 broken. Relevancy, between 0.22 and 0.88 healthy and 0.00 broken. The healthy relevancy numbers look poor for a dull reason. Sixty-word chunks carry sentences the question did not ask for, and relevancy counts every one. They are still above zero everywhere. Three metrics, ten of ten.

The generator pair did something stranger. Faithfulness, 1.00 healthy and 1.00 broken, zero of ten. Answer Relevancy, 0.97 healthy and 1.00 broken, zero of ten. The broken retriever tied on one metric and won the other. Here is what happened. The broken retriever handed the generator three chunks about the wrong subject, and the generator wrote a polite refusal every time. Both generator metrics rewarded it, and they were right to. The generator did nothing wrong. Its scores just have nothing to say about the retriever, and on the docs team's dashboard they are the only scores there are.

One run is a lead, not a finding. The judge's scores move between runs. So the separation has to hold twice before we call it a signature, the pattern of scores a broken retriever leaves. We score the same twenty cases again. Nothing about the assistant changes and no answer is regenerated. Only the judge runs again. This time we also print the broken answers themselves, next to their generator scores, to see what a 1.00 was awarded to.

In [10]:
# The same cases, scored again. Only the judge is re-run.
before = judge_calls()
t0 = time.time()
PANEL[2] = await score_panel(CASES)
RUN_CALLS[2] = judge_calls() - before
panel_end = judge_calls()
print(f"run 2: {RUN_CALLS[2]} judge calls, {time.time() - t0:.0f} s\n")

print(f"{'metric':<22}{'run 1':>7}{'run 2':>7}   largest move of any one score between runs")
held_on_every_question = []
for name in METRICS:
    run_1 = broken_below_healthy(PANEL[1], name)
    run_2 = broken_below_healthy(PANEL[2], name)
    largest_move = 0.0
    for config in CASES:
        for question in DOC_QUESTIONS:
            move = abs(PANEL[1][config][name][question["id"]].score - PANEL[2][config][name][question["id"]].score)
            largest_move = max(largest_move, move)
    print(f"{name:<22}{run_1:>5}/{len(DOC_QUESTIONS)}{run_2:>5}/{len(DOC_QUESTIONS)}   {largest_move:.2f}")
    if run_1 == len(DOC_QUESTIONS) and run_2 == len(DOC_QUESTIONS):
        held_on_every_question.append(name)
print(f"\nbroken below healthy on every question, on both runs: {held_on_every_question}")

print("\nthe broken configuration's answers, scored by the generator pair on both runs")
for question in DOC_QUESTIONS:
    result = RESULTS["broken"][question["id"]]
    faithful = f"{PANEL[1]['broken']['Faithfulness'][question['id']].score:.2f} {PANEL[2]['broken']['Faithfulness'][question['id']].score:.2f}"
    relevant = f"{PANEL[1]['broken']['Answer Relevancy'][question['id']].score:.2f} {PANEL[2]['broken']['Answer Relevancy'][question['id']].score:.2f}"
    print(f"  {question['id']}  faithfulness {faithful}   answer relevancy {relevant}   {result['answer'][:90]}")


run 2: 300 judge calls, 275 s

metric                  run 1  run 2   largest move of any one score between runs
Contextual Precision     10/10   10/10   0.00
Contextual Recall        10/10   10/10   0.33
Contextual Relevancy     10/10   10/10   0.13
Faithfulness              0/10    0/10   0.11
Answer Relevancy          0/10    0/10   0.00

broken below healthy on every question, on both runs: ['Contextual Precision', 'Contextual Recall', 'Contextual Relevancy']

the broken configuration's answers, scored by the generator pair on both runs
  Q-01  faithfulness 1.00 1.00   answer relevancy 1.00 1.00   Based on the provided documentation, there is no information on how to verify webhook sign
  Q-02  faithfulness 1.00 1.00   answer relevancy 1.00 1.00   Based on the provided documentation, there is no information on how to create a charge.
  Q-03  faithfulness 1.00 1.00   answer relevancy 1.00 1.00   Based on the provided documentation, there is no information about the rate limit in tes

It held. Three hundred more calls, 275 seconds, and precision, recall and relevancy put broken below healthy on ten of ten questions both times. Faithfulness on zero, twice. Answer Relevancy on zero, twice. The judge did move. The largest single score movement was 0.33, on Contextual Recall, then 0.13 on Contextual Relevancy and 0.11 on Faithfulness. Precision and Answer Relevancy did not move at all. Recall and relevancy moved and still separated every time, because 0.00 sits a long way below the lowest healthy score.

Now the refusals at the bottom. Ten of them, and every one scored 1.00 on both generator metrics on both runs. A refusal is faithful by construction. It makes no claim the chunks can contradict. And the judge behind Answer Relevancy read "there is no information" as a direct answer to the question, ten times out of ten. Neither metric can see that the chunks were wrong, because neither checks the chunks against the reference. That is the trio's job.

## Two failures, and the metric that saw both

The break above was blunt. A retriever that returns the wrong chunks for every question trips three alarms and misses two. The traps are quieter and much closer to what actually happens on a docs site. In both, the retriever returns chunks that are about the question. The generator answers from them faithfully. And a developer who follows the answer gets hurt.

The webhook trap is the retrieval failure from the opening scene. The deprecated page ranked first. At top-k 3 the third chunk happened to be the deprecated page's own last sentence, the one that names the deprecation. The generator read it. So we run the same question at top-k 2, one chunk fewer, and print the answer in full. On a docs store of thousands of chunks, the second half of a deprecated page does not sit at rank three. Two chunks is the closer model of a real index.

The charge trap looks like the opposite kind of failure. The retriever returns the charge page, exactly the page anyone would expect, and the answer is faithful to it. The rule that a retry without an `Idempotency-Key` creates a second charge sits on a page the question never named. We ask one thing of the answer. Does it mention idempotency. And we run the question once more at top-k 4, to find out where the rule was hiding.

In [11]:
NARROW = build_assistant(inline_retriever(2))      # the same retriever, one chunk fewer
WIDER = build_assistant(inline_retriever(4))       # the same retriever, one chunk more
webhook_narrow = ask(NARROW, QUESTION_BY_ID["Q-01"])
charge = RESULTS["healthy"]["Q-02"]
charge_wider = ask(WIDER, QUESTION_BY_ID["Q-02"])
IDEMPOTENCY = re.compile(r"idempotency", re.I)

print("Webhook question, top-k 2")
for rank, chunk in enumerate(webhook_narrow["retrieved"], 1):
    print(f"rank {rank}  {chunk['part']:<26}status {chunk['status']}")
print("-" * 78)
print(webhook_narrow["answer"])
print("-" * 78)
narrow_v1 = bool(V1_SCHEME.search(webhook_narrow["answer"]))
narrow_v2 = bool(V2_SCHEME.search(webhook_narrow["answer"]))
print(f"top-k 2: names the v1 scheme {narrow_v1}, names the v2 scheme {narrow_v2}")
print(f"top-k 3: names the v1 scheme {names_v1}, names the v2 scheme {names_v2}   (the suite's answer, printed earlier)")

print("\n\nCharge question, top-k 3, the suite's answer")
for rank, chunk in enumerate(charge["retrieved"], 1):
    print(f"rank {rank}  {chunk['part']:<26}status {chunk['status']}")
print("-" * 78)
print(charge["answer"])
print("-" * 78)
charge_mentions = bool(IDEMPOTENCY.search(charge["answer"]))
wider_mentions = bool(IDEMPOTENCY.search(charge_wider["answer"]))
wider_parts = ", ".join(chunk["part"] for chunk in charge_wider["retrieved"])
print(f"top-k 3: answer mentions Idempotency-Key {charge_mentions}")
print(f"top-k 4: answer mentions Idempotency-Key {wider_mentions}   retrieved {wider_parts}")
for sentence in sentences(charge_wider["answer"].replace("\n", " ")):
    if IDEMPOTENCY.search(sentence):
        print(f"  top-k 4 says: {sentence.strip()}")
print(f"\napp model calls: {len(USAGE)}   judge calls: {judge_calls()}")


Webhook question, top-k 2
rank 1  webhook-signatures-v1#1   status deprecated
rank 2  webhook-signatures-v2#1   status current
------------------------------------------------------------------------------
Based on the supplied documentation, signature verification depends on which header your webhook request contains:

### For `X-Halcyon-Signature`:
1. Get the raw request body exact bytes **before** any JSON parsing.
2. Compute a hex-encoded HMAC-SHA1 of the raw request body using your endpoint's signing secret (which starts with `whsec_`).
3. Compare the computed value with the `X-Halcyon-Signature` header using a constant-time comparison.
4. Reject the event if the values differ.

### For `Halcyon-Signature`:
* The header format is `t=<unix timestamp>,v2=<hex digest>`.
* The payload to verify/sign is constructed as: `<unix timestamp>.<raw request body>`.

*(Note: The provided documentation does not include further details on the specific hashing algorithm or comparison steps for the

Here is the answer the developer in the opening scene got. At top-k 2, rank 1 is the deprecated SHA1 chunk again and rank 2 is the first chunk of the current page. The deprecation notice is nowhere. The answer gives the SHA1 scheme in full. Raw body, HMAC-SHA1, the `whsec_` secret, a constant-time comparison, reject on mismatch. For the current scheme it gives the header format and the payload, then apologizes that the documentation does not include the hashing algorithm. Names v1 True, names v2 False. A developer reading that has exactly one complete recipe, and it is the one Halcyon stopped signing with in March. Every sentence of it is faithful to the chunks. She would ship it. She did.

At top-k 3 the same question named both schemes. The difference between a half-fired trap and a fully fired one was one chunk of k, on a corpus of thirty chunks.

Then the charge question. Rank 1 is the quickstart, ranks 2 and 3 are the charge page. The answer lists the endpoint, the three required parameters and the optional ones, and it is a good answer. It never mentions idempotency. A developer who follows it and retries on a timeout creates a second charge. That is the incident behind the money tickets in the corpus.

At top-k 4 the fourth chunk is `api-conventions#2`, the one with the rule, and the answer grows a line. An `Idempotency-Key` header, a unique client-generated string, a UUID v4 recommended. The rule was one rank away. The retriever stopped one chunk short. The plan for this post called the charge trap a failure the retriever could not have prevented. The output says it could have, here, with one more chunk.

Now the table this post exists for. Three rows. The webhook trap as the suite ran it at top-k 3, the same trap at top-k 2, and the charge trap. Five columns, one per metric, each with both runs. The top-k 2 case was not in the suite, so we score it twice first. Under the table, the verdicts that name each problem.

In [12]:
narrow_case = test_case(webhook_narrow)
NARROW_RUNS = {}    # NARROW_RUNS[run][metric name] -> measured metric, the top-k 2 webhook case scored twice
before = judge_calls()
for run in (1, 2):
    NARROW_RUNS[run] = {}
    for name, metric_class in METRICS.items():
        metric = metric_class(model=judge, threshold=None)
        await metric.a_measure(narrow_case, _show_indicator=False)
        NARROW_RUNS[run][name] = metric
print(f"judge calls for the top-k 2 case, two runs: {judge_calls() - before}\n")

TABLE = {}          # TABLE[trap row][metric name][run] -> score
TABLE["webhook, top-k 3"] = {}
TABLE["webhook, top-k 2"] = {}
TABLE["charge, top-k 3"] = {}
for name in METRICS:
    TABLE["webhook, top-k 3"][name] = {1: PANEL[1]["healthy"][name]["Q-01"].score, 2: PANEL[2]["healthy"][name]["Q-01"].score}
    TABLE["webhook, top-k 2"][name] = {1: NARROW_RUNS[1][name].score, 2: NARROW_RUNS[2][name].score}
    TABLE["charge, top-k 3"][name] = {1: PANEL[1]["healthy"][name]["Q-02"].score, 2: PANEL[2]["healthy"][name]["Q-02"].score}

header = f"{'trap':<20}"
for name in METRICS:
    header += f"{name:>22}"
print(header + "     (run 1 / run 2)")
for row_name, row in TABLE.items():
    line = f"{row_name:<20}"
    for name in METRICS:
        line += f"{row[name][1]:>13.2f} / {row[name][2]:.2f}"
    print(line)

print("\nbelow 1.00 on both runs:")
for row_name, row in TABLE.items():
    below = []
    for name in METRICS:
        if row[name][1] < 1.0 and row[name][2] < 1.0:
            below.append(name)
    print(f"  {row_name:<20}{below}")

print("\nthe verdict that names the problem, run 1")
webhook_recall = PANEL[1]["healthy"]["Contextual Recall"]["Q-01"]
narrow_recall = NARROW_RUNS[1]["Contextual Recall"]
charge_recall = PANEL[1]["healthy"]["Contextual Recall"]["Q-02"]
for label, metric in (("webhook, top-k 3", webhook_recall), ("webhook, top-k 2", narrow_recall), ("charge, top-k 3", charge_recall)):
    for verdict in metric.verdicts:
        if verdict.verdict == "no":
            print(f"  {label:<20}recall says no: {verdict.reason}")

print("\nContextual Precision's verdict on the deprecated chunk at rank 1, both runs")
for run in (1, 2):
    suite_verdict = PANEL[run]["healthy"]["Contextual Precision"]["Q-01"].verdicts[0]
    narrow_verdict = NARROW_RUNS[run]["Contextual Precision"].verdicts[0]
    print(f"  run {run}  top-k 3: {suite_verdict.verdict}   top-k 2: {narrow_verdict.verdict}   {narrow_verdict.reason}")


judge calls for the top-k 2 case, two runs: 28

trap                  Contextual Precision     Contextual Recall  Contextual Relevancy          Faithfulness      Answer Relevancy     (run 1 / run 2)
webhook, top-k 3             1.00 / 1.00         0.50 / 0.50         1.00 / 1.00         1.00 / 1.00         1.00 / 1.00
webhook, top-k 2             1.00 / 1.00         0.50 / 0.50         1.00 / 1.00         1.00 / 1.00         1.00 / 1.00
charge, top-k 3              1.00 / 1.00         0.50 / 0.50         0.71 / 0.75         1.00 / 0.89         1.00 / 1.00

below 1.00 on both runs:
  webhook, top-k 3    ['Contextual Recall']
  webhook, top-k 2    ['Contextual Recall']
  charge, top-k 3     ['Contextual Recall', 'Contextual Relevancy']

the verdict that names the problem, run 1
  webhook, top-k 3    recall says no: While parts of the sentence are supported by the 1st, 2nd, and 3rd nodes, the requirement to 'reject the event if the timestamp is more than 300 seconds old' is not mentioned 

Twenty-eight calls for the top-k 2 case, and then the table. It reads best down the columns.

Contextual Precision: 1.00 in every cell. Answer Relevancy: 1.00 in every cell. Contextual Relevancy: 1.00 on both webhook rows, 0.71 and 0.75 on the charge row. Faithfulness: 1.00 everywhere except one run on the charge row, at 0.89. Contextual Recall: 0.50 in every cell.

Below 1.00 on both runs, the webhook rows list one metric, Contextual Recall. The charge row lists two, recall and relevancy. But relevancy's 0.71 and 0.75 sit inside the range it gave the healthy suite in the panel, where sixty-word chunks carry unasked-for sentences. Its miss is chunk length. Recall's miss is the trap, and the verdicts say so in plain words. On the top-k 3 webhook row the judge names the 300-second rule as missing. On the top-k 2 row it names both, HMAC-SHA1 where the reference says HMAC-SHA256, and the 300-second rule. On the charge row it names the `Idempotency-Key` header, the UUID and the retry behaviour.

And at the bottom, Contextual Precision marked the deprecated chunk useful on both runs at both top-k values. Each time it cited the `whsec_` secret and the constant-time comparison. Two phrases shared between a dead page and the right answer were enough.

So the two traps did not fail on opposite sides of the split. They failed on the same side. Both were retrieval failures, the deprecated page ranked first and the needed page ranked fourth, and one metric saw both. It was the one reference-based retriever metric. The four referenceless metrics, two retriever and two generator, scored both traps at or near 1.00 on both runs. Precision and relevancy were fooled because a deprecated page is about the question. Faithfulness and Answer Relevancy were not fooled at all. They graded the generator, and the generator was innocent both times. The split worked. It said retrieval, twice, and it was right twice. The number that said it came from the only metric that knew the right answer.

## What changes when retrieval is a real vector database

An inline list of chunks is not how a docs site retrieves, and the fix does not live in the list. A real site keeps its chunks in a vector database, a store that indexes vectors for nearest-neighbour search and keeps metadata beside each one. DeepEval documents six of them under its vector database integrations. We use Chroma, for two reasons. It runs inside the notebook, and DeepEval's synthesizer requires it to generate goldens from documents, so the dependency pays for a later post.

The wiring is short. The collection takes the vectors we already paid for, the chunk texts, and a metadata record per chunk holding `slug`, `status` and `updated`. `chroma_retriever()` embeds the question and asks the collection for the nearest chunks. Its `where` argument is what this post has been pointing at since the chunk that split. It filters on metadata before the nearest neighbours are taken. We build two assistants on the same collection, one without a filter and one with `where={"status": "current"}`. Both get the webhook question. The filtered one also gets the charge question, to be honest about which trap the filter does not touch. Then the filtered webhook case is scored with all five metrics.

In [13]:
chroma = chromadb.Client(chromadb.config.Settings(anonymized_telemetry=False))   # in-memory; a docs site would persist
collection = chroma.create_collection(name="halcyon-docs", metadata={"hnsw:space": "cosine"})

ids = []
embeddings = []
documents = []
metadatas = []
for chunk in CHUNKS:
    ids.append(chunk["part"])
    embeddings.append(chunk["vector"])                      # the vectors we already paid for
    documents.append(chunk["text"])
    metadatas.append({"slug": chunk["slug"], "status": chunk["status"], "updated": chunk["updated"]})
collection.add(ids=ids, embeddings=embeddings, documents=documents, metadatas=metadatas)
print(f"chroma collection: {collection.count()} chunks, metadata fields {sorted(metadatas[0])}")

def chroma_retriever(top_k, where=None):
    """A retriever over the Chroma collection. where filters on chunk metadata before the nearest neighbours are taken."""
    def retrieve_chunks(question_text):
        query_vector = embedder.embed_text(question_text)
        hits = collection.query(query_embeddings=[query_vector], n_results=top_k, where=where,
                                include=["documents", "metadatas", "distances"])
        retrieved = []
        for part, distance, text, metadata in zip(hits["ids"][0], hits["distances"][0], hits["documents"][0], hits["metadatas"][0]):
            retrieved.append({"part": part, "slug": metadata["slug"], "status": metadata["status"],
                              "updated": metadata["updated"], "text": text, "similarity": 1 - distance})
        return retrieved
    return retrieve_chunks

UNFILTERED = build_assistant(chroma_retriever(TOP_K))
FILTERED = build_assistant(chroma_retriever(TOP_K, where={"status": "current"}))

for label, assistant in (("no filter", UNFILTERED), ("where status = current", FILTERED)):
    result = ask(assistant, QUESTION_BY_ID["Q-01"])
    print(f"\nwebhook question, {label}")
    for rank, chunk in enumerate(result["retrieved"], 1):
        print(f"  rank {rank}  {chunk['similarity']:.3f}  {chunk['part']:<26}{chunk['status']}")
    print(f"  names the v1 scheme {bool(V1_SCHEME.search(result['answer']))}, names the v2 scheme {bool(V2_SCHEME.search(result['answer']))}")
    if label != "no filter":
        webhook_filtered = result

charge_filtered = ask(FILTERED, QUESTION_BY_ID["Q-02"])
filtered_parts = ", ".join(chunk["part"] for chunk in charge_filtered["retrieved"])
print(f"\ncharge question, where status = current: retrieved {filtered_parts}")
print(f"  answer mentions Idempotency-Key {bool(IDEMPOTENCY.search(charge_filtered['answer']))}")

filtered_case = test_case(webhook_filtered)
before = judge_calls()
print("\nthe filtered webhook case, all five metrics")
for name, metric_class in METRICS.items():
    metric = metric_class(model=judge, threshold=None)
    await metric.a_measure(filtered_case, _show_indicator=False)
    print(f"  {name:<22}{metric.score:.2f}   top-k 3 without the filter, run 1: {TABLE['webhook, top-k 3'][name][1]:.2f}")
print(f"judge calls: {judge_calls() - before}")


chroma collection: 30 chunks, metadata fields ['slug', 'status', 'updated']



webhook question, no filter
  rank 1  0.851  webhook-signatures-v1#1   deprecated
  rank 2  0.837  webhook-signatures-v2#1   current
  rank 3  0.799  webhook-signatures-v1#2   deprecated
  names the v1 scheme True, names the v2 scheme True



webhook question, where status = current
  rank 1  0.837  webhook-signatures-v2#1   current
  rank 2  0.767  changelog-2026-03#1       current
  rank 3  0.748  webhook-signatures-v2#2   current
  names the v1 scheme False, names the v2 scheme True



charge question, where status = current: retrieved quickstart#2, charges-create#1, charges-create#2
  answer mentions Idempotency-Key False

the filtered webhook case, all five metrics


  Contextual Precision  0.83   top-k 3 without the filter, run 1: 1.00


  Contextual Recall     1.00   top-k 3 without the filter, run 1: 0.50


  Contextual Relevancy  1.00   top-k 3 without the filter, run 1: 1.00


  Faithfulness          1.00   top-k 3 without the filter, run 1: 1.00


  Answer Relevancy      1.00   top-k 3 without the filter, run 1: 1.00
judge calls: 15


Without the filter, Chroma returns the same three chunks in the same order the inline retriever did. Similarities 0.851, 0.837 and 0.799, the deprecated chunk first, and the answer names both schemes as before. Nothing about a real database fixes this on its own. With `where={"status": "current"}` the deprecated chunks are gone before the neighbours are taken. Rank 1 is the current page's first chunk and rank 2 the changelog. Rank 3 is the current page's second chunk, the one with the algorithm and the timestamp rule. The answer names the v2 scheme and never mentions SHA1. That is the answer the developer should have got in the first place.

The charge question, through the same filter, retrieves the same three chunks as before and the answer still omits the header. The filter fixes the trap that metadata describes. It cannot fix the one that sits on a current page the question did not name.

Scored, the filtered webhook case moves where recall said it would. Contextual Recall 1.00, from 0.50. Faithfulness, Answer Relevancy and Contextual Relevancy 1.00, unchanged. And then one metric went down while the retrieval got better. Contextual Precision 0.83, from 1.00. Over three chunks that score can only be yes, no, yes. The judge marked the middle one, the changelog, not useful. So the third chunk is scored at 2 of 3, and the average of 1 and 2 of 3 is 0.83. Precision's 1.00 before the filter was a score for three useful chunks in a good order, one of them dead. Its 0.83 after is a score for the right page with the changelog in the middle. Neither number says which retrieval a developer would rather have. Recall does.

In a real docs store the value in the filter comes from the session, each account's API version. A deprecated page is then served only to the accounts still pinned to it. The idea does not change. The synthesizer that writes goldens from documents in a later post needs this collection and this embedder. Both were built to last.

## What teams get wrong

**Reading faithfulness as correctness.** Faithfulness scored 1.00 on the SHA1 recipe and 1.00 on the charge answer that omits the header. It scored 1.00 on all ten refusals from a retriever returning the wrong pages. It measures whether the answer contradicts the chunks, and only that. A made-up fact the chunks say nothing about gets an idk verdict, which counts as faithful. `penalize_ambiguous_claims=True` changes that, at the price of every judge misread becoming a failure. When the chunks are wrong, faithful means faithfully wrong. The docs team's green dashboard over a bad index is the scene this post opened with, and this notebook reproduced it three times.

**Running Contextual Recall without ground truth.** Recall was the only metric that saw either trap. It saw them because the reference answer knew the algorithm and knew the header. Fill `expected_output` with whatever is at hand, a previous answer or a page summary, and recall will grade the retriever against that instead. The previous post showed it raise with no reference at all. The quiet failure is a reference that is wrong. It costs the same and returns a number. And on live traffic there is no reference, so the one metric that saw the traps here has nothing to read in production.

**Tuning the prompt when the retriever is the problem.** The generator was innocent in every failure here. The refusals, the SHA1 recipe, the missing header. A team that reads the wrong answers and rewrites the system prompt is tuning the half that works. The retriever trio said retrieval every time, on both runs. The fix was a metadata filter and one more chunk of k. No prompt changed anywhere in this notebook.

The ledger. Every judge call the wrapper saw, thinking tokens included, against DeepEval's own counter for the two panel runs. Then the assistant's calls and the embedding calls, which no earlier post had.

In [14]:
# Our ledger: every judge call the wrapper saw, thinking tokens included.
judge_in = 0
judge_visible = 0
judge_thinking = 0
for usage, temperature in JUDGE_USAGE:
    judge_in += usage.prompt_token_count
    judge_visible += usage.candidates_token_count
    judge_thinking += usage.thoughts_token_count or 0
judge_out = judge_visible + judge_thinking
judge_cost = judge_in * PRICE_IN + judge_out * PRICE_OUT
temperatures = sorted(set(temperature for usage, temperature in JUDGE_USAGE))

# DeepEval's own counter for the two panel runs, against our ledger for the same calls.
deepeval_in = 0
deepeval_cost = 0
for run in PANEL:
    for config in PANEL[run]:
        for name in PANEL[run][config]:
            for metric in PANEL[run][config][name].values():
                deepeval_in += metric.input_tokens
                deepeval_cost += metric.evaluation_cost
panel_in = 0
panel_out = 0
for usage, temperature in JUDGE_USAGE[panel_start:panel_end]:
    panel_in += usage.prompt_token_count
    panel_out += usage.candidates_token_count + (usage.thoughts_token_count or 0)
panel_cost = panel_in * PRICE_IN + panel_out * PRICE_OUT

# The assistant's own calls, recorded by the setup cell, and the embedding calls, recorded by embed().
app_in = 0
app_out = 0
for usage in USAGE:
    app_in += usage.get("input_tokens", 0)
    app_out += usage.get("output_tokens", 0)
app_cost = app_in * PRICE_IN + app_out * PRICE_OUT
embed_texts_sent = 0
embed_tokens = 0
for usage in EMBED_USAGE:
    embed_texts_sent += usage["texts"]
    embed_tokens += usage["tokens"]
embed_cost = embed_tokens * PRICE_EMBED

per_case = ", ".join(f"{name} {calls}" for name, calls in CALLS_PER_CASE.items())
print(f"judge calls      {judge_calls()}   retried after a 5xx: {len(JUDGE_RETRIES)}   temperature sent: {temperatures}")
print(f"judge calls per case at top-k {TOP_K}: {per_case}")
print(f"judge tokens     {judge_in} in / {judge_out} out   (thinking inside 'out': {judge_thinking}, {100 * judge_thinking / judge_out:.0f}%)")
print(f"judge cost       ${judge_cost:.4f}")
print(f"two panel runs   {panel_end - panel_start} calls   ours ${panel_cost:.4f}   DeepEval's counter ${deepeval_cost:.4f}, "
      f"{panel_cost / deepeval_cost:.1f} times less   (input tokens: DeepEval {deepeval_in}, ours {panel_in})")
print(f"app calls        {len(USAGE)}   tokens {app_in} in / {app_out} out   cost ${app_cost:.4f}")
print(f"embed calls      {len(EMBED_USAGE)}   texts {embed_texts_sent}   tokens {embed_tokens}   cost ${embed_cost:.4f}")
print(f"total            {judge_calls() + len(USAGE) + len(EMBED_USAGE)} model calls   ${judge_cost + app_cost + embed_cost:.4f}   "
      f"{time.time() - T0:.0f} s since the setup cell")


judge calls      666   retried after a 5xx: 0   temperature sent: [0.0]
judge calls per case at top-k 3: Contextual Precision 2, Contextual Recall 2, Contextual Relevancy 4, Faithfulness 4, Answer Relevancy 3
judge tokens     306942 in / 417446 out   (thinking inside 'out': 349988, 84%)
judge cost       $1.7956
two panel runs   600 calls   ours $1.5849   DeepEval's counter $0.4321, 3.7 times less   (input tokens: DeepEval 274533, ours 274533)
app calls        25   tokens 6918 in / 13203 out   cost $0.0547
embed calls      26   texts 55   tokens 2256   cost $0.0003
total            717 model calls   $1.8507   848 s since the setup cell


Six hundred and sixty-six judge calls, none retried, every one at temperature 0.0, the value DeepEval put there. Per case at top-k 3: two calls for precision, two for recall, four for faithfulness and three for answer relevancy. Relevancy spends four, one per chunk plus the reason, so it grows with k. The judge sent 306,942 tokens in and got 417,446 out, and 349,988 of those were thinking, 84 percent. The judge cost $\$1.7956$ at the prices we set. The two panel runs were 600 calls where DeepEval's counter and ours saw the same requests. Ours says $\$1.5849$ and DeepEval's says $\$0.4321$, 3.7 times less. Its input count, 274,533, matched ours to the token. The gap is thinking tokens it does not see, the same shape the two earlier metric posts found. The assistant's 25 calls cost $\$0.0547$. Twenty-six embedding calls, 55 texts, 2,256 tokens, $\$0.0003$. Embeddings are a rounding error. The whole notebook cost $\$1.8507$ and ran 848 seconds from the setup cell. Most of both went on the 600 panel calls, because ten questions, two configurations, five metrics and two runs multiply.

## What Halcyon has now, and what it still cannot do

The fingerprint matched. Every number above came from the same pages and tickets as the three posts before it, plus a question set with its own fingerprint.

Halcyon now has a Docs Assistant. A two-node graph, retrieve then generate, where the generator is the node the triage assistant already used. It has ten developer questions with reference answers. It has a retriever scored apart from the generator, with a signature for a broken one, read twice. It has both traps fired and scored. The deprecated page ranked first, the missing header rule one rank away, and a table that says one metric saw both. And it has a Chroma collection with the page metadata beside each chunk. Its filter would have saved the developer in the opening scene a weekend.

What Halcyon still cannot do is watch any of this once it ships. Every score above came from a notebook, on questions we wrote, with references we wrote. Nobody is scoring the answers developers get today, and the reference-based metric that saw both traps has no reference on live traffic. A later post on production monitoring takes that up. Nor can Halcyon catch the charge trap without a reference. The rule that every money-moving request carries an `Idempotency-Key` is a house rule, and the house rule Halcyon has is the previous post's GEval. One sentence, and a judge that rewrites its own steps on every run. The next post writes criteria properly.

Three rules we are taking forward. We score the retriever and the generator apart, and read which one moved. We keep version currency in metadata and filter on it at retrieval time, because chunking strips it from the text. And when a score is 1.00, we ask which text it was checked against. A faithful answer to the wrong page scores the same as a faithful answer to the right one.

### Resources

- DeepEval: RAG evaluation guide, the retriever and generator split: https://deepeval.com/guides/guides-rag-evaluation
- DeepEval: Contextual Precision metric: https://deepeval.com/docs/metrics-contextual-precision
- DeepEval: Contextual Recall metric: https://deepeval.com/docs/metrics-contextual-recall
- DeepEval: Contextual Relevancy metric: https://deepeval.com/docs/metrics-contextual-relevancy
- DeepEval: Faithfulness metric: https://deepeval.com/docs/metrics-faithfulness
- DeepEval: using custom embedding models: https://deepeval.com/guides/guides-using-custom-embedding-models
- DeepEval: Chroma integration: https://deepeval.com/integrations/vector-databases/chroma
- Gemini API: embeddings and task types: https://ai.google.dev/gemini-api/docs/embeddings
- Gemini API pricing: https://ai.google.dev/gemini-api/docs/pricing
- Companion repository with this notebook and its pyproject.toml: https://github.com/tuhinsharma121/ai-playground/blob/main/eval/deepeval/03-notebook.ipynb